In [1]:
import findspark
findspark.init("/opt/spark")

In [2]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, HashingTF, IDF, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [3]:
spark = SparkSession.builder.appName("Turkish News Text Classification").master("local[*]") \
.config("spark.driver.memory", "4g") \
.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/07/28 08:12:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
sc = spark.sparkContext

In [5]:
from pyspark import SparkFiles
p0 = "https://github.com/erkansirin78/datasets/raw/master/turkish_news/train_parquet/part-00000-b3e150fd-453b-467e-9194-c72fc884e331-c000.snappy.parquet"
# p1 = "https://github.com/erkansirin78/datasets/raw/master/turkish_news/train_parquet/part-00001-b3e150fd-453b-467e-9194-c72fc884e331-c000.snappy.parquet"
# p2 = "https://github.com/erkansirin78/datasets/raw/master/turkish_news/train_parquet/part-00002-b3e150fd-453b-467e-9194-c72fc884e331-c000.snappy.parquet"
# p3 = "https://github.com/erkansirin78/datasets/raw/master/turkish_news/train_parquet/part-00003-b3e150fd-453b-467e-9194-c72fc884e331-c000.snappy.parquet"
# p4 = "https://github.com/erkansirin78/datasets/raw/master/turkish_news/train_parquet/part-00004-b3e150fd-453b-467e-9194-c72fc884e331-c000.snappy.parquet"
# p5 = "https://github.com/erkansirin78/datasets/raw/master/turkish_news/train_parquet/part-00005-b3e150fd-453b-467e-9194-c72fc884e331-c000.snappy.parquet"
# p6 = "https://github.com/erkansirin78/datasets/raw/master/turkish_news/train_parquet/part-00006-b3e150fd-453b-467e-9194-c72fc884e331-c000.snappy.parquet"
# p7 = "https://github.com/erkansirin78/datasets/raw/master/turkish_news/train_parquet/part-00007-b3e150fd-453b-467e-9194-c72fc884e331-c000.snappy.parquet"
sc.addFile(p0)
# sc.addFile(p1)
# sc.addFile(p2)
# sc.addFile(p3)
# sc.addFile(p4)
# sc.addFile(p5)
# sc.addFile(p6)
# sc.addFile(p7)

df = spark.read.parquet("file:///"+SparkFiles.get("*"))

In [6]:
df.cache()

DataFrame[content: string, category: bigint]

In [7]:
df.count()

27360

In [8]:
df.limit(5).toPandas()

,content,category
0,Esmerler daha geç yaşlanıyor Doç. Dr. Zerrin Ö...,2
1,"Dijitalde markanız, sanal gerçeklik cihazları ...",6
2,Antepli Chico yu ceza durdurdu Sezon başında A...,5
3,"ŞEKER HASTALIĞI DA, KANSER DE İYİLEŞİR Ünlü K...",8
4,"İCRAAT ZAMAN Iiiı Kongre bitti, dosya tamamlan...",5


In [9]:
tokenizer = Tokenizer(inputCol="content", outputCol="words")

In [10]:
df2 = tokenizer.transform(df)

In [11]:
df2.limit(5).toPandas()

,content,category,words
0,Esmerler daha geç yaşlanıyor Doç. Dr. Zerrin Ö...,2,"[esmerler, daha, geç, yaşlanıyor, doç., dr., z..."
1,"Dijitalde markanız, sanal gerçeklik cihazları ...",6,"[dijitalde, markanız,, sanal, gerçeklik, cihaz..."
2,Antepli Chico yu ceza durdurdu Sezon başında A...,5,"[antepli, chico, yu, ceza, durdurdu, sezon, ba..."
3,"ŞEKER HASTALIĞI DA, KANSER DE İYİLEŞİR Ünlü K...",8,"[, şeker, hastaliği, da,, kanser, de, i̇yi̇leş..."
4,"İCRAAT ZAMAN Iiiı Kongre bitti, dosya tamamlan...",5,"[i̇craat, zaman, iiiı, kongre, bitti,, dosya, ..."


In [12]:
hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures")

In [13]:
df3 = hashingTF.transform(df2)

In [14]:
df3.limit(5).toPandas()

,content,category,words,rawFeatures
0,Esmerler daha geç yaşlanıyor Doç. Dr. Zerrin Ö...,2,"[esmerler, daha, geç, yaşlanıyor, doç., dr., z...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,"Dijitalde markanız, sanal gerçeklik cihazları ...",6,"[dijitalde, markanız,, sanal, gerçeklik, cihaz...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,Antepli Chico yu ceza durdurdu Sezon başında A...,5,"[antepli, chico, yu, ceza, durdurdu, sezon, ba...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,"ŞEKER HASTALIĞI DA, KANSER DE İYİLEŞİR Ünlü K...",8,"[, şeker, hastaliği, da,, kanser, de, i̇yi̇leş...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,"İCRAAT ZAMAN Iiiı Kongre bitti, dosya tamamlan...",5,"[i̇craat, zaman, iiiı, kongre, bitti,, dosya, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [15]:
df3.printSchema()

root
 |-- content: string (nullable = true)
 |-- category: long (nullable = true)
 |-- words: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- rawFeatures: vector (nullable = true)



In [16]:
idf = IDF(inputCol="rawFeatures", outputCol="features")

In [17]:
df4 = idf.fit(df3).transform(df3)

In [18]:
df4.limit(5).toPandas()

24/07/28 08:14:28 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB


,content,category,words,rawFeatures,features
0,Esmerler daha geç yaşlanıyor Doç. Dr. Zerrin Ö...,2,"[esmerler, daha, geç, yaşlanıyor, doç., dr., z...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,"Dijitalde markanız, sanal gerçeklik cihazları ...",6,"[dijitalde, markanız,, sanal, gerçeklik, cihaz...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,Antepli Chico yu ceza durdurdu Sezon başında A...,5,"[antepli, chico, yu, ceza, durdurdu, sezon, ba...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,"ŞEKER HASTALIĞI DA, KANSER DE İYİLEŞİR Ünlü K...",8,"[, şeker, hastaliği, da,, kanser, de, i̇yi̇leş...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,"İCRAAT ZAMAN Iiiı Kongre bitti, dosya tamamlan...",5,"[i̇craat, zaman, iiiı, kongre, bitti,, dosya, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [19]:
df5 = df4.withColumnRenamed("category", "label")

In [20]:
df5.limit(5).toPandas()

24/07/28 08:14:29 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB


,content,label,words,rawFeatures,features
0,Esmerler daha geç yaşlanıyor Doç. Dr. Zerrin Ö...,2,"[esmerler, daha, geç, yaşlanıyor, doç., dr., z...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,"Dijitalde markanız, sanal gerçeklik cihazları ...",6,"[dijitalde, markanız,, sanal, gerçeklik, cihaz...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,Antepli Chico yu ceza durdurdu Sezon başında A...,5,"[antepli, chico, yu, ceza, durdurdu, sezon, ba...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,"ŞEKER HASTALIĞI DA, KANSER DE İYİLEŞİR Ünlü K...",8,"[, şeker, hastaliği, da,, kanser, de, i̇yi̇leş...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,"İCRAAT ZAMAN Iiiı Kongre bitti, dosya tamamlan...",5,"[i̇craat, zaman, iiiı, kongre, bitti,, dosya, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [21]:
# Logistic Regression Model
lr = LogisticRegression(maxIter=10, regParam=0.01)

In [22]:
df6 = lr.fit(df5.sample(fraction=0.01)).transform(df5)

24/07/28 08:14:32 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 08:14:34 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
24/07/28 08:14:37 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 08:15:12 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 08:15:50 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 08:16:28 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 08:17:06 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 08:17:45 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 08:18:23 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 08:19:00 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 08:19:40 WARN DAGScheduler: Broadcasting large task binary with size 4.1 MiB
24/07/28 08:20:19 WARN DAGSchedul

In [23]:
df6.limit(5).toPandas()

24/07/28 08:22:02 WARN DAGScheduler: Broadcasting large task binary with size 8.2 MiB


,content,label,words,rawFeatures,features,rawPrediction,probability,prediction
0,Esmerler daha geç yaşlanıyor Doç. Dr. Zerrin Ö...,2,"[esmerler, daha, geç, yaşlanıyor, doç., dr., z...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[1.9455083987182458, 0.2918046062875552, -0.48...","[0.37442958902635926, 0.07164332434338849, 0.0...",0.0
1,"Dijitalde markanız, sanal gerçeklik cihazları ...",6,"[dijitalde, markanız,, sanal, gerçeklik, cihaz...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-1.3793227887401205, -0.4213666933484327, -0....","[0.00019280151818331149, 0.0005025109103977742...",6.0
2,Antepli Chico yu ceza durdurdu Sezon başında A...,5,"[antepli, chico, yu, ceza, durdurdu, sezon, ba...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[1.6372577698995041, 0.10361349340140985, -0.6...","[0.1413079314390116, 0.030486901840688762, 0.0...",5.0
3,"ŞEKER HASTALIĞI DA, KANSER DE İYİLEŞİR Ünlü K...",8,"[, şeker, hastaliği, da,, kanser, de, i̇yi̇leş...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[1.2161699079561434, 0.725043204847836, -0.470...","[0.1021899226938152, 0.06253374718761254, 0.01...",8.0
4,"İCRAAT ZAMAN Iiiı Kongre bitti, dosya tamamlan...",5,"[i̇craat, zaman, iiiı, kongre, bitti,, dosya, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-2.111852454501418, -0.28292719020914747, 1.6...","[0.000972430217763242, 0.006055508191497431, 0...",5.0


In [24]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

In [25]:
evaluator = MulticlassClassificationEvaluator()

In [26]:
evaluator.evaluate(df6)

24/07/28 08:24:26 WARN DAGScheduler: Broadcasting large task binary with size 8.2 MiB


0.5377682607557661

In [27]:
evaluator.getMetricName()

'f1'